In [1]:
import os
import json
import pandas as pd
import traceback
import PyPDF2
from pathlib import Path
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableConfig
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks.manager import get_openai_callback

from dotenv import load_dotenv
load_dotenv()

/tmp/ipykernel_10062/2278009214.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks.manager import get_openai_callback


True

In [2]:
llm = ChatOpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1",
    model="openai/gpt-oss-120b",
    temperature=0.5,
)

In [3]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    }, 
}

In [4]:
TEMPLATE="""
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to create a quiz of 
{number} multiple choice questions for {subject} students in {tone} tone. Make sure the
questions are not repeated and check all the questions to be confirming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}
"""

In [5]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [6]:
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

In [7]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject}
students. You need to evaluate the complexity of the question and give a complete analysis of
the quiz. Only use at max 50 words for complexity if the quiz is not at per with the cognitive 
and analytical abilities of the students. Update the quiz questions which needs to be changed
and change the tone such that it perfectly fits the students' ability.
Quiz MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [8]:
quiz_evaluation_prompt=PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [9]:
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [10]:
generate_evaluate_chain = (
    RunnablePassthrough()
    .assign(quiz=quiz_chain)
    .assign(review=review_chain)
)

In [11]:
file_path = Path("/workspaces/MCQ_Generator/data.txt")

if not file_path.is_file():
    local_candidate = Path("data.txt")
    if local_candidate.is_file():
        file_path = local_candidate
    else:
        raise FileNotFoundError(
            f"Could not find the input text file at {file_path} or "
            f"{local_candidate.resolve()}. Update file_path to the location of data.txt."
        )

TEXT = file_path.read_text(encoding="utf-8")

NUMBER = 5
SUBJECT = "machine learning"
TONE = "simple"

In [12]:
response = generate_evaluate_chain.invoke(
    {
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON, ensure_ascii=False),
    }
)

print("QUIZ")
print(response["quiz"])
print("\\nREVIEW")
print(response["review"])


QUIZ
{
  "1": {
    "mcq": "Who coined the term \"machine learning\" in 1959?",
    "options": {
      "a": "Geoffrey Hinton",
      "b": "Arthur Samuel",
      "c": "Tom M. Mitchell",
      "d": "Alan Turing"
    },
    "correct": "Arthur Samuel"
  },
  "2": {
    "mcq": "What was the earliest machine‑learning program created by Arthur Samuel?",
    "options": {
      "a": "A program that played chess",
      "b": "A program that calculated the chance of winning in checkers",
      "c": "A program that recognized handwritten digits",
      "d": "A program that translated languages"
    },
    "correct": "A program that calculated the chance of winning in checkers"
  },
  "3": {
    "mcq": "Which theory, introduced by Donald Hebb in 1949, helped shape many modern ML algorithms?",
    "options": {
      "a": "Hebbian theory",
      "b": "Bayesian theory",
      "c": "Support‑vector theory",
      "d": "Evolutionary theory"
    },
    "correct": "Hebbian theory"
  },
  "4": {
    "mcq": 

In [13]:
quiz = response.get("quiz")
quiz = json.loads(quiz)

In [15]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [16]:
quiz_table_data

[{'MCQ': 'Who coined the term "machine learning" in 1959?',
  'Choices': 'a: Geoffrey Hinton | b: Arthur Samuel | c: Tom M. Mitchell | d: Alan Turing',
  'Correct': 'Arthur Samuel'},
 {'MCQ': 'What was the earliest machine‑learning program created by Arthur Samuel?',
  'Choices': 'a: A program that played chess | b: A program that calculated the chance of winning in checkers | c: A program that recognized handwritten digits | d: A program that translated languages',
  'Correct': 'A program that calculated the chance of winning in checkers'},
 {'MCQ': 'Which theory, introduced by Donald Hebb in 1949, helped shape many modern ML algorithms?',
  'Choices': 'a: Hebbian theory | b: Bayesian theory | c: Support‑vector theory | d: Evolutionary theory',
  'Correct': 'Hebbian theory'},
 {'MCQ': 'Which neural network won the ImageNet competition in 2012 and sparked widespread use of deep learning?',
  'Choices': 'a: VGGNet | b: ResNet | c: AlexNet | d: Inception',
  'Correct': 'AlexNet'},
 {'MCQ

In [24]:
quiz=pd.DataFrame(quiz_table_data)
quiz.to_csv("machine_learning.csv", index=False)

In [17]:
datetime.now()

datetime.datetime(2026, 10, 11, 6, 43, 4, 293534)